# Welcome to the [30 Days of ML competition](https://www.kaggle.com/c/30-days-of-ml/overview)!

<center><img src='https://storage.googleapis.com/kaggle-media/Images/30_Days_ML_Hero.png' width='480' height="480" ></center>

#### This notebook will go through an extensive analysis to view and explore the `30 Days of ML` competition data, there will be some "well, maybe a lot of" redundancies but come on that's why "extensive" is in the name, also this notebook is a work in progress so there is a lot to be added later.
<h4><font color='darkred'>Please note that this work you're about to see here is not memory efficient at all but this notebook was meant to be extensive to explore different approaches also the data is not that big</font></h4>


### The notebook will have the following sections:
 #### - [Data Imports and initial exploring](#load_data).
 #### - [exploratory data analysis "EDA"](#EDA).
 #### - [Cleaning and feature Engineering starter](#ft_eng).
 #### - [XGBOOST starter](#xgb). 


<h2><center>Let's dive right in!</center></h2>
<center><img src="https://64.media.tumblr.com/801c1d244924a60dfae54af4924dda9e/033f51f7de7447be-d9/s1280x1920/6343410ac221ab8bc34c005c19b2718d8f1e96a3.gifv"></center>

In [ ]:
# imports
import gc
import time
from IPython.display import Image, display
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 40)
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from termcolor import colored
%matplotlib inline
sns.set_style('darkgrid')
import plotly.express as px
# import plotly.graph_objects as go

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import mutual_info_regression
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.model_selection import train_test_split as split
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans 
from sklearn.model_selection import GridSearchCV as Grid
import xgboost as xgb
from sklearn.metrics import (mean_absolute_error, r2_score,mean_squared_error)

<a id='load_data'></a>
# Loading Data and exploring main statistics 

We set `index_col=0` in the code cell below to use the `id` column to index the DataFrame.

In [ ]:
train = pd.read_csv("../input/30-days-of-ml/train.csv", index_col=0)
test = pd.read_csv("../input/30-days-of-ml/test.csv", index_col=0)

# Preview the data
train.head()

In [ ]:
train.describe()

In [ ]:
train.info()

In [ ]:
train.isnull().sum()

In [ ]:
features = train.drop(['target'], axis=1)

#### To easily distinguish them we will extract the name of the columns of different data types

In [ ]:
num_col = list(train.select_dtypes(include='float64').columns)
cat_cols = list(train.select_dtypes(include='object').columns)
num_col.remove('target')
print('Number of numerical columns is:',colored(len(num_col),'green'),
      '\nNumber of categorical columsn is:',colored(len(cat_cols),'green'))

#### Assuring that test data and whether or not it has the same columns as the train

In [ ]:
list(test.columns) == list(features.columns)

In [ ]:
test.describe()

In [ ]:
test.info()

In [ ]:
test.isnull().sum()

#### Checking if test categorical unqiue values are all subsets of their train peers

In [ ]:
lis = []
for i in features[cat_cols].columns:
    test_vals = set(test[i].unique())
    train_vals = set(features[i].unique())
    lis.append(test_vals.issubset(train_vals))

print(colored(all(lis),'green'))

<center><img src="https://c.tenor.com/EGhxbE0xUvIAAAAC/we-good-thumbs-up.gif"></center>

> #### Well Data seems to be pretty much preprocessed "I mean it's synthetic after all" so no nulls exists and also even if features are anonymous they are ordered in a proper easy to the eye way.

<a id='EDA'></a>
# EDA

#### Now we start exploring the data set , mainly there are three main groups of variables to explore as follows:
 - Categorical columns.
 - Continuous "numerical" columns.
 - Target.

### Categorical variables

#### Number of categorical unique values

In [ ]:
fig = plt.figure(figsize=(10,5))
sns.barplot(y=train[cat_cols].nunique().values, x=train[cat_cols].nunique().index, color='blue', alpha=.5)
plt.xticks(rotation=0)
plt.title('Number of categorical unique values',fontsize=16);

> #### Most of columns seems to have few categorical unique values except cat9 column.

#### Categorical features unique values count

In [ ]:
fig = plt.figure(figsize=(26,10))
grid =  gridspec.GridSpec(2,5,figure=fig,hspace=.2,wspace=.2)
n =0
for i in range(2):
    for j in range(5):
        ax = fig.add_subplot(grid[i, j])
        order = list(train['cat'+str(n)].value_counts().index)
        sns.countplot(data= train, x='cat'+str(n),ax=ax, alpha =0.8,order=order,palette='viridis')
        ax.set_title('cat'+str(n),fontsize=14)
        ax.set_xlabel('')
        ax.set_ylabel('')
        n += 1
fig.suptitle('Train categorical features unique values count', fontsize=16,y=.93);

In [ ]:
fig = plt.figure(figsize=(26,10))
grid =  gridspec.GridSpec(2,5,figure=fig,hspace=.2,wspace=.2)
n =0
for i in range(2):
    for j in range(5):
        ax = fig.add_subplot(grid[i, j])
        order = list(test['cat'+str(n)].value_counts().index)
        sns.countplot(data= test, x='cat'+str(n),ax=ax, alpha =0.8,order=order,palette='viridis')
        ax.set_title('cat'+str(n),fontsize=14)
        ax.set_xlabel('')
        ax.set_ylabel('')
        n += 1
fig.suptitle('Test categorical features unique values count', fontsize=16,y=.93);

>#### Distribution of unique values in test and train data looks pretty much similar.

#### Distribution of categorical features unique values and target

In [ ]:
fig = plt.figure(figsize=(26,10))
grid =  gridspec.GridSpec(2,5,figure=fig,hspace=.2,wspace=.2)
n =0
for i in range(2):
    for j in range(5):
        ax = fig.add_subplot(grid[i, j])
        sns.barplot(data= train, y = 'target', x='cat'+str(n),ax=ax, alpha =.6,ci=95, color= 'darkblue',dodge=False )
        ax.set_title('cat'+str(n),fontsize=14)
        ax.set_xlabel('')
        ax.set_ylabel('')
        n += 1
fig.suptitle('Distribution of categorical features unique values and target', fontsize=16,y=.93);

> #### It looks like on the surface there is no clear relation between target and any of categorical variables' values.

#### KDE plot of target with categorical features

In [ ]:
fig = plt.figure(figsize=(26,10))
grid =  gridspec.GridSpec(2,5,figure=fig,hspace=.2,wspace=.2)
n =0
for i in range(2):
    for j in range(5):
        ax = fig.add_subplot(grid[i, j])
        sns.kdeplot(data = train, x = 'target', hue = 'cat'+str(n),ax=ax, alpha =.7, fill=False)
        ax.set_title('cat'+str(n),fontsize=14)
        ax.set_xlabel('')
        ax.set_ylabel('')
        n += 1
fig.suptitle('KDE plot of train target with categorical features', fontsize=16,y=.93);

> #### This plot kinda agrees with previous one but it looks like the KDE of some categorical values are pretty much flat compared to other value.

#### Violin plot of target with categorical features

In [ ]:
fig = plt.figure(figsize=(30,10))
grid =  gridspec.GridSpec(2,5,figure=fig,hspace=.2,wspace=.2)
n =0
for i in range(2):
    for j in range(5):
        ax = fig.add_subplot(grid[i, j])
        sns.violinplot(data = train, y = 'target', x = 'cat'+str(n),ax=ax, alpha =.7, fill=True,palette='viridis')
        ax.set_title('cat'+str(n),fontsize=14)
        ax.set_xlabel('')
        ax.set_ylabel('')
        n += 1
fig.suptitle('Violin plot of target with categorical features', fontsize=16,y=.93);

<center><img src="https://www.memesmonkey.com/images/memesmonkey/3f/3f459b9e453447e0bedded09eba42df6.jpeg"></center>

> #### This plot is a continuation to the KDE plot and it pretty muh agrees with it but we can also notice target outliers.

### Numerical Variables

#### Box plot of numerical columns

In [ ]:
v0 = sns.color_palette(palette='viridis').as_hex()[0]
fig = plt.figure(figsize=(18,6))
sns.boxplot(data=train[num_col], color=v0,saturation=.5);
plt.xticks(fontsize= 14)
plt.title('Box plot of train numerical columns', fontsize=16);

In [ ]:
fig = plt.figure(figsize=(18,6))
sns.boxplot(data=test[num_col], color=v0,saturation=.5);
plt.xticks(fontsize= 14)
plt.title('Box plot of test numerical columns', fontsize=16);

> #### Numerical Data seems to be kinda normalized with few outliers appearing in the box plot Also test numerical data seems to looks like the train ones.

#### Histograms of numerical features

In [ ]:
fig = plt.figure(figsize=(28,10))#,constrained_layout=True)
grid =  gridspec.GridSpec(2, 7, figure= fig, hspace= .3, wspace= .2)
n =0
for i in range(2):
    for j in range(7):
        ax = fig.add_subplot(grid[i, j])
        sns.histplot(data= train, x='cont'+str(n),ax=ax, alpha =.6, color= 'darkblue',kde=True)
        ax.set_title('cont'+str(n),fontsize=14)
        ax.set_xlabel('')
        ax.set_ylabel('')
        n += 1
        
fig.suptitle('Histograms of train numerical features', fontsize=16,y=.93);

In [ ]:
fig = plt.figure(figsize=(28,10))#,constrained_layout=True)
grid =  gridspec.GridSpec(2, 7, figure= fig, hspace= .3, wspace= .2)
n =0
for i in range(2):
    for j in range(7):
        ax = fig.add_subplot(grid[i, j])
        sns.histplot(data= test, x='cont'+str(n),ax=ax, alpha =.6, color= 'darkblue',kde=True)
        ax.set_title('cont'+str(n),fontsize=14)
        ax.set_xlabel('')
        ax.set_ylabel('')
        n += 1
        
fig.suptitle('Histograms of test numerical features', fontsize=16,y=.93);

> #### Histograms of numerical data show a desperation of values with what look like multinomial distributions, also column cont1 seems to have some areas where the distribution becomes kinda discrete and again test numerical data seems to be similar to train numerical data.

#### Scatter plot with Pearson coefficient of correlation of numerical features with target

In [ ]:
fig = plt.figure(figsize=(24,8))#,constrained_layout=True)
grid =  gridspec.GridSpec(2, 7, figure= fig, hspace= .3, wspace= .2)
n =0
for i in range(2):
    for j in range(7):
        ax = fig.add_subplot(grid[i, j])
        sns.scatterplot(data= train, x='cont'+str(n), y='target',ax=ax, alpha =.4, color= 'darkblue' )
        ax.set_title('cont{}\n corr with target = {}%'.format(str(n),round(100*(train['cont'+str(n)].corr(train.target)),3)),fontsize=12)
        ax.set_xlabel('')
        ax.set_ylabel('')
        n += 1
        
fig.suptitle('Scatter plot of numerical features with target', fontsize=18,y=.98)
fig.text(0.11,0.5, "Target", ha="center", va="center", rotation=90, fontsize=18);

#### Zooming on the correlation between numerical variables and target.

In [ ]:
train.corr()['target'][:-1].plot.barh(figsize=(8,6),alpha=.6,color='darkblue')
plt.xlim(-.075,.075);
plt.xticks([-0.065, -0.05 , -0.025,  0.   ,  0.025,  0.05 ,  0.065],
           [str(100*i)+'%' for i in [-0.065, -0.05 , -0.025,  0.   ,  0.025,  0.05 ,  0.065]],fontsize=12)
plt.title('Correlation between target and numerical variables',fontsize=14);

> #### It's clear tat there isn't any clear relation between numerical variables and target.

#### Now Exploring correlation between all numerical variables.

#### First we get a correlation grid of all numercial variables and target

In [ ]:
train.corr().style.background_gradient(cmap='viridis')

#### Showing a grid scatter plots to investigate the numbers shown above

In [ ]:
sns.pairplot(train[num_col], corner=True, diag_kind='kde');

> #### Well again as seen in the correlation grid there isn't strong relationship between variables, also these variables have kind of multinomial distributions.

### Target data


 #### separating the target (`y`) from the training features (which we assign to `features`).

In [ ]:
y = train['target']

#### don't mind me,  just a sanity check 

In [ ]:
(y.index == features.index).all()

<img src='https://c.tenor.com/_f5-QsYtLt0AAAAC/crazy-jim-carrey.gif'>

In [ ]:
plt.figure(figsize=(12,6))
sns.histplot(y, bins = 150)
plt.title('Histogram of target data', fontsize= 20)
plt.ylabel('');

In [ ]:
plt.figure(figsize=(8,6))
sns.boxplot(y= y, width=.2)
plt.title('Box plot of target data', fontsize= 20)
plt.ylabel('');

> #### Target seems to have lots of extremes outliers

#### Standardizing target

#### Now we try to explore if standardizing target will yield anything intersting

In [ ]:
scaler = StandardScaler()
y_scaled = pd.DataFrame(scaler.fit_transform(pd.DataFrame(y)))
y_scaled.columns = [y.name]
y_scaled.index = y.index

In [ ]:
fig = plt.figure(figsize=(28,11))#,constrained_layout=True)
grid =  gridspec.GridSpec(2, 7, figure= fig, hspace= .3, wspace= .2)
n =0
for i in range(2):
    for j in range(7):
        ax = fig.add_subplot(grid[i, j])
        sns.scatterplot(data= train, y='cont'+str(n), x=y_scaled.target,ax=ax, alpha =.5, color= 'darkblue' )
        ax.set_title('cont{}\n corr with target = {}% '.format(str(n),round(100*(train['cont'+str(n)].corr(train.target)),3)),fontsize=12)
        ax.set_xlabel('')
        ax.set_ylabel('')
        n += 1
        
fig.suptitle('Scatter plot of numerical features with target', fontsize=18,y=.94)
fig.text(0.11,0.5, "Target", ha="center", va="center", rotation=90, fontsize=18);

> #### Nothing impressive but xticks but again we notice outliers.

#### Exploring target data main statistics 

In [ ]:
y.describe().iloc[1:].plot.barh(color=v0,alpha=.5,figsize=(12,5))
plt.title('Target data statistics',fontsize=16)
plt.yticks(fontsize=14)
plt.xticks(np.arange(0,10.8,.5));

In [ ]:
np.percentile(y,(25,75))

> #### Minimum value seems to be way below the avrage.

#### Box/violin plot of target data with different percentiles limits darwn

In [ ]:
plt.figure(figsize=(12,6))
sns.boxplot(x=y, width=.4);
plt.axvline(np.percentile(y,.1), label='.1%', c='orange', linestyle=':', linewidth=3)
plt.axvline(np.percentile(y,.5), label='.5%', c='darkblue', linestyle=':', linewidth=3)
plt.axvline(np.percentile(y,1), label='1%', c='green', linestyle=':', linewidth=3)
plt.axvline(np.percentile(y,99), label='99%', c='gold', linestyle=':', linewidth=3)
plt.axvline(np.percentile(y,99.9), label='99.9%', c='red', linestyle=':', linewidth=3)
plt.legend()
plt.title('Box plot of target data', fontsize=16)
plt.xticks(np.arange(0,10.8,.5));

In [ ]:
plt.figure(figsize=(12,6))
sns.violinplot(x=y, width=.4);
plt.axvline(np.percentile(y,.1), label='.1%', c='orange', linestyle=':', linewidth=3)
plt.axvline(np.percentile(y,.5), label='.5%', c='darkblue', linestyle=':', linewidth=3)
plt.axvline(np.percentile(y,1), label='1%', c='green', linestyle=':', linewidth=3)
plt.axvline(np.percentile(y,99), label='99%', c='gold', linestyle=':', linewidth=3)
plt.axvline(np.percentile(y,99.9), label='99.9%', c='red', linestyle=':', linewidth=3)
plt.legend()
plt.title('Violin plot of target data', fontsize=16)
plt.xticks(np.arange(0,10.8,.5));

> #### Seems like outliers need to be handled

<a id='ft_eng'></a>
# Cleaning and feature Engineering 

### Exploring Clustering and PCA
#### Mainly there are two way to do this 
- First PCA and then clustering
- Clustering and then doing PCA

#### It also worth mentioning that there is two ways of using clusters as features
- using cluster label (in short one columns with each number representing the cluster" 
- using clusters labels ( adding k number of columns representing the distance to the centroid of each cluster"

#### First we do Clustring "using num_col only"

In [ ]:
clus = KMeans(n_clusters=8, max_iter=800)
kmean_no_pca= clus.fit_predict(train[num_col])

train_clus = train.copy()
train_clus['cluster'] = kmean_no_pca
train_clus['cluster'] = train_clus['cluster'].astype('object')

test_clus = test.copy()
test_clus['cluster'] = clus.predict(test[num_col]).astype('object')

In [ ]:
plt.figure(figsize=(19,6))
sns.violinplot(data=train_clus, y='target', x='cluster',palette='viridis');
plt.axhline(np.percentile(y,.1), label='.1%', c='red', linestyle=':', linewidth=3)
plt.axhline(np.percentile(y,.5), label='.5%', c='orange', linestyle=':', linewidth=3)
plt.axhline(np.percentile(y,1), label='1%', c='green', linestyle=':', linewidth=3)
plt.axhline(np.percentile(y,99), label='99%', c='gold', linestyle=':', linewidth=3)
plt.axhline(np.percentile(y,99.9), label='99.9%', c='darkblue', linestyle=':', linewidth=3)
plt.legend(ncol=3,loc= (.55,.01))
plt.title('Violin plot of target data "with clusters"', fontsize=18)
plt.yticks(np.arange(0,10.8,.5));

In [ ]:
plt.figure(figsize=(12,6))
sns.kdeplot(data=train_clus, x='target', hue='cluster',palette='viridis',alpha=.3)
plt.title('KDE plot of target data "with clusters"', fontsize=18);

In [ ]:
fig = plt.figure(figsize=(32,24))#,constrained_layout=True)
grid =  gridspec.GridSpec(5, 2, figure= fig, hspace= .2, wspace= .05)
n =0
for i in range(5):
    for j in range(2):
        ax = fig.add_subplot(grid[i, j])
        sns.pointplot(data=train_clus, y='target',x= 'cluster',hue='cat'+str(n), ax=ax, palette='viridis', ci='sd', capsize=.05,join=True,dodge=.6 )
        ax.set_title('cat{}'.format(str(n)),fontsize=16)
        ax.set_xlabel('')
        ax.set_ylabel('')
        ax.set_ylim(6.8,10.1)
        ax.legend(loc='upper left',ncol=20)
        n += 1
        
fig.suptitle('Pointplot of Clusters, Target and Categorical features', fontsize=20,y=.90)
fig.text(0.11,0.5, "Target", ha="center", va="center", rotation=90, fontsize=18);

In [ ]:
fig = plt.figure(figsize=(12,32))#,constrained_layout=True)
grid =  gridspec.GridSpec(7, 2, figure= fig, hspace= .2, wspace= .05)
n =0
for i in range(7):
    for j in range(2):
        ax = fig.add_subplot(grid[i, j])
        sns.scatterplot(data=train_clus, y='target', x='cont'+str(n), hue= 'cluster', ax=ax, palette='viridis', alpha=.6 )
        ax.set_title('cont{}'.format(str(n)),fontsize=16)
        ax.set_xlabel('')
        ax.set_ylabel('')
        ax.legend(loc='lower left',ncol=20)
        n += 1
        
fig.suptitle('Scatter plot of Target, Numerical and Cluster features', fontsize=20,y=.90)
fig.text(0.08,0.5, "Target", ha="center", va="center", rotation=90, fontsize=18);

> #### Clustering seems like it wont add anything, but there is some intersting relation noticed with cat 6 and cat 7

#### Now Exploring PCA

In [ ]:
pca = PCA(n_components=6)
columns = ['pca_%i' % i for i in range(1,7)]

df_pca = pd.DataFrame(pca.fit_transform(train[num_col]), columns=columns, index=train.index)
test_pca = pd.DataFrame(pca.transform(test[num_col]), columns=columns, index=test.index)
df_pca = train.drop(columns=num_col).merge(df_pca, left_index=True, right_index=True)
test_pca = test.drop(columns=num_col).merge(test_pca, left_index=True, right_index=True)

clus_pca = KMeans(n_clusters=3, max_iter=1000)
kmean_pca= clus_pca.fit_predict(df_pca[columns])
kmeans_test_pca = clus_pca.predict(test_pca[columns])

df_pca_clus = df_pca.copy()
test_pca_clus = test_pca.copy()
test_pca_clus['cluster'] = kmeans_test_pca
test_pca_clus['cluster'] = test_pca_clus['cluster'].astype('object') 
df_pca_clus['cluster'] = kmean_pca
df_pca_clus['cluster'] = df_pca_clus['cluster'].astype('object')

#### Getting the correcorrelation between targets and principal components

In [ ]:
df_pca.corr().style.background_gradient(cmap='viridis')

In [ ]:
df_pca[columns].hist(layout=(3,2),bins=80, figsize=(16,10), alpha=.6);

In [ ]:
fig = plt.figure(figsize=(12,16))#,constrained_layout=True)
grid =  gridspec.GridSpec(3, 2, figure= fig, hspace= .2, wspace= .05)
n =1
for i in range(3):
    for j in range(2):
        ax = fig.add_subplot(grid[i, j])
        sns.scatterplot(data=df_pca_clus, y='target', x='pca_'+str(n), hue= 'cluster', ax=ax, palette='viridis', alpha=.6 )
        ax.set_title('pca_{}'.format(str(n)),fontsize=16)
        ax.set_xlabel('')
        ax.set_ylabel('')
        ax.legend(loc='lower left',ncol=20)
        n += 1

fig.suptitle('Scatter plot of Target, principal components and Cluster features', fontsize=20,y=.92)
fig.text(0.08,0.5, "Target", ha="center", va="center", rotation=90, fontsize=18);

#### Exploring the PCA loading
##### [Taken from Feature Engineering course lesson 5 of 6](https://www.kaggle.com/ryanholbrook/principal-component-analysis)

In [ ]:
loadings = pd.DataFrame(
    pca.components_.T,  # transpose the matrix of loadings
    columns=columns,  # so the columns are the principal components
    index=num_col,  # and the rows are the original features
)
loadings.style.background_gradient(cmap='viridis')

### Now we get the mutual information scores

In [ ]:
# mi_scores_pca = mutual_info_regression(df_pca[columns], y, random_state=0)
# gc.collect()
# mi_scores_pca = pd.Series(mi_scores_pca, name="MI_Scores_pca", index=df_pca[columns].columns)
# gc.collect()
# mi_scores_pca = mi_scores_pca.sort_values(ascending=False)
# mi_scores_pca.to_dict()
pca_dict = {'pca_1': 0.011851056820256112,'pca_2': 0.0030647695643661876,'pca_5': 0.0023570309909342058,
 'pca_3': 0.0011543896910328755,'pca_4': 0.0007548110478801107,'pca_6': 0.0}
mi_scores_pca = pd.DataFrame(pca_dict.values(), index=pca_dict.keys(),columns=['MI_Scores_pca'] )
mi_scores_pca.style.background_gradient(cmap='viridis')

> #### Seems like this data is not a walk in the park at all

<center><img src='https://c.tenor.com/ZFc20z8DItkAAAAd/facepalm-really.gif'></center>

#### Exploring different data trunctation

In [ ]:
print('train target .1%: ', np.percentile(train.target,.1))
print('TRUNCATING',end='')
print('.' ,end ='')
time.sleep(.5)
      
min_01= np.percentile(train.target,.1)
train_01_trunc = train[train.target > min_01]
print('.' ,end ='')
time.sleep(.5)

min_05 = np.percentile(train.target,.5)
train_05_trunc = train[train.target > min_05]
print('.' ,end ='')
time.sleep(.5)

min_1 = np.percentile(train.target,1)
train_1_trunc = train[train.target > min_1]
print('.' ,end ='')
time.sleep(.5)


max_99 = np.percentile(train.target,99)
train_1_99_trunc = train[(train.target > min_1) & (train.target < max_99)]
print('\nDone!' ,end ='')

In [ ]:
features = train.drop(['target'], axis=1)
features_clus = train_clus.drop(['target'], axis=1)

y_01_trunc = train_01_trunc['target']
features_01_trunc = train_01_trunc.drop(['target'], axis=1)

y_05_trunc = train_05_trunc['target']
features_05_trunc = train_05_trunc.drop(['target'], axis=1)

y_1_trunc = train_1_trunc['target']
features_1_trunc = train_1_trunc.drop(['target'], axis=1)

y_1_99_trunc = train_1_99_trunc['target']
features_1_99_trunc = train_1_99_trunc.drop(['target'], axis=1)

In [ ]:
fig, (ax1,ax2,ax3,ax4)= plt.subplots(1,4,figsize=(22,5))
plt.subplots_adjust(left=None, bottom=None, right=None, top=.85, wspace=None, hspace=.3)

sns.histplot(y_01_trunc, kde=True, color= v0, alpha=.6,bins=200,ax=ax1)
ax1.set_ylabel('')
ax1.set_xlabel('')
ax1.set_title('.1 %',fontsize=16)
# ax1.set_xticks(np.arange(5.5,11,.5));

sns.histplot(y_05_trunc, kde=True, color= v0, alpha=.6,bins=200,ax=ax2)
ax2.set_ylabel('')
ax2.set_xlabel('')
ax2.set_title('.5 %',fontsize=16)
# ax2.set_xticks(np.arange(5.5,11,.5));

sns.histplot(y_1_trunc, kde=True, color= v0, alpha=.6,bins=200,ax=ax3)
ax3.set_ylabel('')
ax3.set_xlabel('')
ax3.set_title('1 %',fontsize=16)
# ax3.set_xticks(np.arange(5.5,11,.5));

sns.histplot(y_1_99_trunc, kde=True, color= v0, alpha=.6,bins=200,ax=ax4)
ax4.set_ylabel('')
ax4.set_xlabel('')
ax4.set_title('1:99 %',fontsize=16)
# ax4.set_xticks(np.arange(5.5,11,.5))

fig.suptitle('Histplot of target data after removing outliers', fontsize= 20);

In [ ]:
fig, (ax1, ax2, ax3, ax4)= plt.subplots(1,4,figsize=(18,6))

sns.boxplot(y=y_01_trunc, color= v0,ax=ax1, width=.3)
ax1.set_ylabel('')
ax1.set_xlabel('')
ax1.set_title('.1 %',fontsize=16)
ax1.set_yticks(np.arange(5.5,11,.5));

sns.boxplot(y=y_05_trunc, color= v0,ax=ax2, width=.3)
ax2.set_ylabel('')
ax2.set_xlabel('')
ax2.set_title('.5 %',fontsize=16)
ax2.set_yticks(np.arange(5.5,11,.5));

sns.boxplot(y=y_1_trunc, color= v0,ax=ax3, width=.3)
ax3.set_ylabel('')
ax3.set_xlabel('')
ax3.set_title('1 %',fontsize=16)
ax3.set_yticks(np.arange(5.5,11,.5));

sns.boxplot(y= y_1_99_trunc, color= v0,ax=ax4, width=.3)
ax4.set_ylabel('')
ax4.set_xlabel('')
ax4.set_title('1:99 %',fontsize=16)
ax4.set_yticks(np.arange(5.5,11,.5));


fig.suptitle('Boxplot of target data after removing outliers', fontsize= 20);

> #### Outliers still exist but we will need to explore model quality to find out whether or not it's possbile to drop more data, it's also worth mentioning that the histogram shows kind of steps-like behavior around some values.

#### Categorical values handling:
#### mainly there are two way to handle those "aside from dropping them!)
- One-hot encoding
- Ordinal encoding

#### we will explore both to find out the better way to handle categorical variables
#### Using the pd.get_dummies to make one hot encoding to categorical data

#### Making interaction columns for cat 6 and cat 7 with cluster

In [ ]:
features_int = train_clus.drop(['target'], axis=1)
features_int['c6_int'] = features_int.cat6 +'_'+ features_int.cluster.astype(str)
features_int['c7_int'] = features_int.cat7 +'_'+ features_int.cluster.astype(str)

test_int = test_clus.copy()
test_int['c6_int'] = test_int.cat6 +'_'+ test_int.cluster.astype(str)
test_int['c7_int'] = test_int.cat7 +'_'+ test_int.cluster.astype(str)

In [ ]:
train_dummies = pd.get_dummies(features)
train_clus_dummies = pd.get_dummies(features_clus)
test_dummies = pd.get_dummies(test)
test_clus_dummies = pd.get_dummies(test_clus)

df_pca_dummies = pd.get_dummies(df_pca)
df_pca_clus_dummies = pd.get_dummies(df_pca_clus)

train_dummies_int = pd.get_dummies(features_int)
test_dummies_int = pd.get_dummies(test_int)

train_trunc_dummies_01 = pd.get_dummies(features_01_trunc)
train_trunc_dummies_05 = pd.get_dummies(features_05_trunc)
train_trunc_dummies_1 = pd.get_dummies(features_1_trunc)
train_trunc_dummies_1_99 = pd.get_dummies(features_1_99_trunc)

In [ ]:
for i in (set(train_dummies_int.columns) - set(test_dummies_int.columns)):
        test_dummies_int[i] = 0

In [ ]:
test_dummies_int = test_dummies_int[train_dummies_int.columns]

In [ ]:
df_pca_dummies.drop(columns='target',inplace=True)
df_pca_clus_dummies.drop(columns='target',inplace=True)

In [ ]:
list(train_dummies_int.columns) == list(test_dummies_int.columns)

#### Checking if we lost any information in truncation

In [ ]:
all(train_trunc_dummies_1_99.columns == train_dummies.columns) == all(test_dummies.columns == train_dummies.columns)

<center><img src='https://c.tenor.com/WRwm-wTN0_8AAAAd/i-just-got-lucky-willie.gif' width='380' height="380"></center>

In [ ]:
## This is Sklearn implmentation of one-hot encoding

# OH_encoder = OneHotEncoder(handle_unknown='ignore', sparse=False)
# OH_cols_train = pd.DataFrame(OH_encoder.fit_transform(features[cat_cols]))
# OH_cols_test = pd.DataFrame(OH_encoder.transform(test[cat_cols]))

# OH_cols_train.index = features.index
# OH_cols_test.index = test.index

# num_X_train = features.drop(cat_cols, axis=1)
# num_X_test = test.drop(cat_cols, axis=1)

# OH_X_train = pd.concat([num_X_train, OH_cols_train], axis=1)
# OH_X_test = pd.concat([num_X_test, OH_cols_test], axis=1)

#### Now we make an ordinal encoding to explore the model quality using it compared to one hot encoding

In [ ]:
OR_encoder = OrdinalEncoder()

OR_cols_train = pd.DataFrame(OR_encoder.fit_transform(features[cat_cols]))
OR_cols_test = pd.DataFrame(OR_encoder.transform(test[cat_cols]))
OR_cols_train_trunc_01 = pd.DataFrame(OR_encoder.transform(features_01_trunc[cat_cols]))
OR_cols_train_trunc_05 = pd.DataFrame(OR_encoder.transform(features_05_trunc[cat_cols]))
OR_cols_train_trunc_1 = pd.DataFrame(OR_encoder.transform(features_1_trunc[cat_cols]))
OR_cols_train_trunc_1_99 = pd.DataFrame(OR_encoder.transform(features_1_99_trunc[cat_cols]))
OR_cols_pca = pd.DataFrame(OR_encoder.transform(df_pca[cat_cols]))


OR_cols_train.index = features.index
OR_cols_test.index = test.index
OR_cols_train_trunc_01.index = features_01_trunc.index
OR_cols_train_trunc_05.index = features_05_trunc.index
OR_cols_train_trunc_1.index = features_1_trunc.index
OR_cols_train_trunc_1_99.index = features_1_99_trunc.index
OR_cols_pca.index = df_pca.index

OR_cols_train.columns = cat_cols
OR_cols_test.columns = cat_cols
OR_cols_train_trunc_01.columns = cat_cols
OR_cols_train_trunc_05.columns = cat_cols
OR_cols_train_trunc_1.columns = cat_cols
OR_cols_train_trunc_1_99.columns = cat_cols
OR_cols_pca.columns = cat_cols

num_X_train = features.drop(cat_cols, axis=1)
num_X_test = test.drop(cat_cols, axis=1)
num_X_train_trunc_01 = features_01_trunc.drop(cat_cols, axis=1)
num_X_train_trunc_05 = features_05_trunc.drop(cat_cols, axis=1)
num_X_train_trunc_1 = features_1_trunc.drop(cat_cols, axis=1)
num_X_train_trunc_1_99 = features_1_99_trunc.drop(cat_cols, axis=1)
num_X_pca = df_pca.drop(cat_cols, axis=1)

OR_X_train = pd.concat([num_X_train, OR_cols_train], axis=1)
OR_X_test = pd.concat([num_X_test, OR_cols_test], axis=1)
OR_X_train_trunc_01 = pd.concat([num_X_train_trunc_01, OR_cols_train_trunc_01], axis=1)
OR_X_train_trunc_05 = pd.concat([num_X_train_trunc_05, OR_cols_train_trunc_05], axis=1)
OR_X_train_trunc_1 = pd.concat([num_X_train_trunc_1, OR_cols_train_trunc_1], axis=1)
OR_X_train_trunc_1_99 = pd.concat([num_X_train_trunc_1_99, OR_cols_train_trunc_1_99], axis=1)
OR_X_pca =  pd.concat([num_X_pca, OR_cols_pca], axis=1)

In [ ]:
OR_X_train_clus = OR_X_train.copy()
OR_X_train_clus['cluster'] = train_clus['cluster'].astype(int)

OR_X_test_clus = OR_X_test.copy()
OR_X_test_clus['cluster'] = test_clus['cluster'].astype(int)

OR_X_train_clus_trunc_01 = OR_X_train_trunc_01.copy()
OR_X_train_clus_trunc_01['cluster'] = train_clus[train.target > min_01]['cluster'].astype(int)

OR_X_train_clus_trunc_05 = OR_X_train_trunc_05.copy()
OR_X_train_clus_trunc_05['cluster'] = train_clus[train.target > min_05]['cluster'].astype(int)

OR_X_train_clus_trunc_1 = OR_X_train_trunc_1.copy()
OR_X_train_clus_trunc_1['cluster'] = train_clus[train.target > min_1]['cluster'].astype(int)


OR_X_train_clus_trunc_1_99 = OR_X_train_trunc_1_99.copy()
OR_X_train_clus_trunc_1_99['cluster'] = train_clus[(train.target > min_1) & (train.target < max_99)]['cluster'].astype(int)

OR_X_pca_clus = OR_X_pca.copy()
OR_X_pca_clus['cluster'] = df_pca_clus['cluster'].astype(int)

#### Adding feature the will that may help the model perform better

In [ ]:
OR_X_train_int = OR_X_train_clus.copy()
OR_X_train_int['c_c6'] = OR_X_train_int.cluster * OR_X_train_int.cat6
OR_X_train_int['c_c7'] = OR_X_train_int.cluster * OR_X_train_int.cat7

OR_X_test_int = OR_X_test_clus.copy()
OR_X_test_int['c_c6'] = OR_X_test_int.cluster * OR_X_test_int.cat6
OR_X_test_int['c_c7'] = OR_X_test_int.cluster * OR_X_test_int.cat7

<center><img src='https://c.tenor.com/CZT9JqoJYW4AAAAC/repetitive-redundant.gif'></center>

In [ ]:
# mi_scores_dummies = mutual_info_regression(train_dummies, y, random_state=0)
# mi_scores_dummies = pd.Series(mi_scores_dummies, name="MI_Scores_dummies", index=train_dummies.columns)
# mi_scores_dummies = mi_scores_dummies.sort_values(ascending=False)
# mi_scores_dummies

In [ ]:

# mi_scores_ordinal = mutual_info_regression(OR_X_train, y, random_state=0)
# mi_scores_ordinal = pd.Series(mi_scores_ordinal, name="MI_Scores_Ordinal", index=OR_X_train.columns)
# mi_scores_ordinal = mi_scores_ordinal.sort_values(ascending=False)
# mi_scores_ordinal

In [ ]:
# fig ,(ax1,ax2) = plt.subplots(2,1,figsize=(28,10))
# plt.subplots_adjust(left=None, bottom=None, right=None, top=.90, wspace=None, hspace=.4)
# sns.barplot(x = mi_scores_dummies.index, y = mi_scores_dummies.values,palette='viridis', ax=ax1)
# sns.barplot(x = mi_scores_ordinal.index, y = mi_scores_ordinal.values,palette='viridis', ax= ax2)
# ax1.tick_params(axis='x',labelrotation=45,labelsize=12)
# ax2.tick_params(axis='x',labelrotation=45,labelsize=12)
# ax1.set_title('Mutual information with target "One hot encoding"',fontsize=18)
# ax2.set_title('Mutual information with target "Ordinal encoding"',fontsize=18)
# fig.suptitle('Mutual information',fontsize=22);

> #### Mutual information seems to agree with pearson correlation as relationship with target seems to be very week.

#### Ratio of features with MI greater than 0

In [ ]:
# print('features with MI greater than 0 "one hot encoding":',colored(str(round(100*(mi_scores_dummies > 0).mean(),4)) + '%','green'))

In [ ]:
# print('features with MI greater than 0 "ordinal ordinal":',colored(str(round(100*(mi_scores_ordinal > 0).mean(),4)) + '%','green'))

<a id='xgb'></a>

# XGBOOST starter

#### We will be using GridSearchCV with XGBRegressor to get closer to the best model parameters.

#### Submissions are scored on the root mean squared error 'RMSE' which is defined as: 
<h3>$${RMSE} = \sqrt{\frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2}$$</h3>

#### Generally We can explore the quality of these approaches
- ordinal encoding
- one hot encoding

#### You might see the following lines of code commented so you can comment it out and explore the different approaches yourself.
#### As a reminder we have the following sets of data to train and explore the quality of their approachs

- train_dummies
- train_dummies with different truncation / clustering
- OR_X_train
- OR_X_train with different truncation/ clustering
#### you can also try different combination of features using MI extracted earlier.

In [ ]:
# from sklearn.metrics import SCORERS
# SCORERS.keys()

#### The following is a parameters grid to explore in grid search cross validation

In [ ]:
# xgbcpars = {'booster': ['gbtree'],
#             'colsample_bytree': [0.7],
#             'eval_metric': ['rmse'],
#             'gamma': [1],
#             'learning_rate': [0.08],#0.07853392035787837],
#             'max_depth': [3],
#             'n_estimators': [4000],
#             'objective': ['reg:squarederror'],
#             'random_state': [0],
#             'reg_alpha': [1.5],
#             'reg_lambda': [1.7549293092194938e-05],
#             'subsample': [0.9],
#             'colsample_bytree': [0.170759104940733]}

#### Now we make a XGBRegressor to build our model

In [ ]:
# reg = xgb.XGBRegressor(tree_method='gpu_hist')

#### Putting all in the grid to start test the different parameters.

In [ ]:
# grid = Grid(estimator=reg, param_grid= xgbcpars,scoring= 'neg_root_mean_squared_error',
#          verbose= 100, cv= 5 , return_train_score= False)
# grid

#### Fitting the model

In [ ]:
OR_X_train_pca1 = OR_X_train.copy()
OR_X_train_pca1['pca'] = df_pca.pca_1
OR_X_train_pca1
OR_X_test_pca1 = OR_X_test.copy()
OR_X_test_pca1['pca'] = test_pca.pca_1

In [ ]:
# grid.fit(OR_X_train_trunc_01, y_01_trunc)

#### Getting the best best estimator

In [ ]:
# grid.best_estimator_

#### Getting the best parameters

In [ ]:
# grid.best_params_

#### Getting the best score

In [ ]:
# grid.best_score_

In [ ]:
# OR_X_train               = -0.718640474135784
# OR_X_train_pca1          = -0.7187306399092321
# OR_X_train_clus_trunc_01 = -0.7097323223769159
# OR_X_train_trunc_01      =  0.709725258182609
# OR_X_train_int           = -0.7187115650146971
# OR_X_train_clus          = -0.7187287224544328
# train_dummies            = -0.71875469941565
# train_dummies_int        = -0.7188511341200838
# train_clus_dummies       = -0.7186358775342236

In [ ]:
# pd.Series(grid.best_estimator_.feature_importances_, index= OR_X_test_int.columns)

#### Now we define a function called get_score to automate the exhaustive search

In [ ]:
# def get_score(L, grid):
#     """
#     this function will fit a GridSearchCV with XGBRegressor
#     input
#     L: a list of tuples containing train and target couples with 0 index for train data and 1 for target
#     grid: parameters that will be used in the search
#     return
#     a dictionary with train data names as keys and list of their scores and best parameters as values
#     """
#     n = 1
#     dic = {}
#     for data in L:              
#         print(colored('Now Trainin Dataset No: {}'.format(n),'green'))
#         reg = xgb.XGBRegressor(tree_method='gpu_hist', nthread= -1,verbosity =0)
#         grid = Grid(estimator=reg, param_grid= xgbcpars,scoring= 'neg_root_mean_squared_error',
#                     verbose= 0, cv= 5 , return_train_score= False)
#         grid.fit(data[0], data[1])
#         dic[n] = [grid.best_score_, grid.best_params_]
#         print(colored('\nFone with Trainin Dataset No: {} '.format(n),'green'))
#         print('==========================')
#         n += 1
#     return dic

In [ ]:
# get_score([(OR_X_train,y), (OR_X_train_int, y)],xgbcpars)

In [ ]:
# df_xgb = pd.concat([pd.DataFrame(grid.cv_results_["params"]),pd.DataFrame(grid.cv_results_["mean_test_score"], columns=["score"])],axis=1)

#### Extracting feature importances

In [ ]:
# imp=pd.DataFrame(grid.best_estimator_.feature_importances_, index= OR_X_train.columns ,columns=('imp',))

In [ ]:
# imp.sort_values(by='imp',ascending=False,inplace=True)

In [ ]:
# imp.imp.plot.bar(figsize=(22,5))
# plt.xticks(rotation=0);

#### Building final model

In [ ]:
model = xgb.XGBRegressor(booster ='gbtree', colsample_bytree=0.170759104940733,
                         eval_metric= 'rmse', gamma= .7,
                         learning_rate= 0.07853392035787837, max_depth= 5,
                         n_estimators= 80000, objective= 'reg:squarederror',
                         random_state= 0, reg_alpha= 1.5,
                         reg_lambda= 1.7549293092194938e-05, subsample= 0.9,
                         tree_method='gpu_hist')

In [ ]:
model.fit(OR_X_train_pca1, y)

In [ ]:
# preds = model.predict(X_valid)
# print('msqr:', mean_squared_error(y_valid,preds),
#       '\nmabe:',1* mean_absolute_error(y_valid,preds),
#       '\nR2:', r2_score(y_valid,preds))

#### Submit to the competition


In [ ]:
# Use the model to generate predictions
predictions = model.predict(OR_X_test_pca1)

# Save the predictions to a CSV file
output = pd.DataFrame({'Id': OR_X_test_pca1.index,
                       'target': predictions})
output.to_csv('submission.csv', index=False);

<h2><center><font color='red'>Upvote Or Else...</font></center></h2>
<center><img src="https://i.imgur.com/Sbpg1MS.gif"></center>